In [1]:
!pip install pyspark

In [32]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, hour, dayofmonth, month, col, when, expr, abs
from pyspark.sql.functions import (col, unix_timestamp)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Lectura Dataframe

In [3]:
spark = SparkSession.builder \
    .appName("NYC Taxi ML") \
    .getOrCreate()

spark

In [4]:
df = spark.read.csv("yellow_tripdata_2016-03.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag| dropoff_longitude| dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       1| 2016-03-01 00:00:00|  2016-03-01 00:07:55|              1|          2.5|-73.97674560546875| 40.76515197753906|       1.0|       

In [5]:
df.count()

12237852

In [6]:
df_sample = df.sample(withReplacement=False, fraction=0.001, seed=42)
df_sample.printSchema()
df_sample.show(5)
df_sample.count()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+-

12262

# Limpieza

In [7]:
df_clean = df_sample
df_clean = df_clean.withColumn(
    "store_and_fwd_flag",
    when(col("store_and_fwd_flag") == "Y", 1)
    .when(col("store_and_fwd_flag") == "N", 0)
    .otherwise(None)
)

# NUMÉRICAS
numeric_cols = [
    "VendorID",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount"
]

for c in numeric_cols:
    df_clean = df_clean.withColumn(c, expr(f"try_cast({c} as double)"))

# NULLS
df_clean = df_clean.dropna(subset=[
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type"
])
df_clean = df_clean.filter(col("fare_amount") > 0)
df_clean = df_clean.filter(col("fare_amount") < 100)

#Dates
df_clean = df_clean.withColumn("pickup_hour", hour("tpep_pickup_datetime"))
df_clean = df_clean.withColumn("pickup_day", dayofmonth("tpep_pickup_datetime"))
df_clean = df_clean.withColumn("pickup_month", month("tpep_pickup_datetime"))

#Duración
df_clean = df_clean.withColumn(
    "trip_duration",
    (unix_timestamp("tpep_dropoff_datetime") -
     unix_timestamp("tpep_pickup_datetime")) / 60
)

#Outliers
df_clean = df_clean.filter(col("trip_duration") > 0)
df_clean = df_clean.filter(col("trip_duration") < 300)  # opcional
df_clean = df_clean.filter(col("trip_distance") > 0)

#Validación
df_clean.printSchema()
df_clean.select("payment_type", "store_and_fwd_flag").distinct().show()

root
 |-- VendorID: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: integer (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_day: integer (nullable = true)
 |-- pickup_month: integer (nullab

# Conjunto de entrenamiento y prueba

In [8]:
feature_cols = [
    "VendorID",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type",
    "pickup_hour",
    "pickup_day",
    "pickup_month",
    "trip_duration"
]
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
df_ml = assembler.transform(df_clean)

In [9]:
df_ml.select(
    "trip_distance",
    "trip_duration",
    "features"
).show(5, truncate=False)

+-------------+------------------+----------------------------------------------------------+
|trip_distance|trip_duration     |features                                                  |
+-------------+------------------+----------------------------------------------------------+
|0.6          |2.75              |[1.0,1.0,0.6,1.0,0.0,1.0,12.0,1.0,3.0,2.75]               |
|0.86         |10.016666666666667|[2.0,4.0,0.86,1.0,0.0,1.0,12.0,1.0,3.0,10.016666666666667]|
|0.8          |13.266666666666667|[1.0,2.0,0.8,1.0,0.0,1.0,12.0,1.0,3.0,13.266666666666667] |
|1.19         |7.383333333333334 |[2.0,1.0,1.19,1.0,0.0,2.0,12.0,1.0,3.0,7.383333333333334] |
|0.52         |8.2               |[2.0,1.0,0.52,1.0,0.0,1.0,12.0,1.0,3.0,8.2]               |
+-------------+------------------+----------------------------------------------------------+
only showing top 5 rows


In [10]:
df_clean.select(feature_cols).printSchema()

root
 |-- VendorID: double (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: integer (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_day: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- trip_duration: double (nullable = true)



In [11]:
df_ml.select("fare_amount").summary().show()

+-------+------------------+
|summary|       fare_amount|
+-------+------------------+
|  count|             12177|
|   mean|12.577695655744437|
| stddev| 9.998862645324914|
|    min|              0.05|
|    25%|               6.5|
|    50%|               9.5|
|    75%|              14.5|
|    max|              96.0|
+-------+------------------+



In [12]:
df_ml.select("features").show(3, truncate=False)

+----------------------------------------------------------+
|features                                                  |
+----------------------------------------------------------+
|[1.0,1.0,0.6,1.0,0.0,1.0,12.0,1.0,3.0,2.75]               |
|[2.0,4.0,0.86,1.0,0.0,1.0,12.0,1.0,3.0,10.016666666666667]|
|[1.0,2.0,0.8,1.0,0.0,1.0,12.0,1.0,3.0,13.266666666666667] |
+----------------------------------------------------------+
only showing top 3 rows


In [13]:
train, test = df_ml.randomSplit([0.8, 0.2], seed=42)

# Random Forest Regressor

In [14]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=10,
    maxDepth=5,
    seed=42
)
model_rf = rf.fit(train)

In [15]:
predictions = model_rf.transform(test)

In [16]:
rmse = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(predictions)

mae = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="mae"
).evaluate(predictions)

r2 = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
).evaluate(predictions)

print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

RMSE: 2.444745018927912
MAE: 1.3560511230366388
R²: 0.9420539965853088


In [17]:
predictions.select(
    "fare_amount",
    "prediction"
).show(10, truncate=False)

+-----------+------------------+
|fare_amount|prediction        |
+-----------+------------------+
|7.0        |7.577447809466254 |
|15.0       |13.990711076551099|
|9.0        |9.237401283878725 |
|6.0        |6.571704890040019 |
|5.0        |6.418964321893567 |
|13.0       |12.193399922589439|
|11.0       |10.201749204135178|
|12.0       |11.001183780963151|
|5.5        |6.279856424473728 |
|10.0       |8.828939382588537 |
+-----------+------------------+
only showing top 10 rows


In [18]:
df_clean.select("fare_amount").summary().show()

+-------+------------------+
|summary|       fare_amount|
+-------+------------------+
|  count|             12177|
|   mean|12.577695655744437|
| stddev| 9.998862645324914|
|    min|              0.05|
|    25%|               6.5|
|    50%|               9.5|
|    75%|              14.5|
|    max|              96.0|
+-------+------------------+



In [19]:
model_rf.featureImportances

SparseVector(10, {0: 0.0001, 1: 0.0007, 2: 0.5244, 3: 0.1685, 4: 0.0005, 5: 0.0022, 6: 0.0021, 7: 0.0015, 9: 0.3001})

In [20]:
for feature, importance in zip(feature_cols, model_rf.featureImportances):
    print(feature, importance)

VendorID 5.3187230122533704e-05
passenger_count 0.0007077817284026341
trip_distance 0.5244214828844369
RatecodeID 0.16848794695002273
store_and_fwd_flag 0.0004899530169740369
payment_type 0.0021882358904654494
pickup_hour 0.0020506809809154995
pickup_day 0.001516817810207674
pickup_month 0.0
trip_duration 0.3000839135084525


In [23]:
predictions.select(
    "fare_amount",
    "prediction"
).withColumn(
    "error",
    abs(col("fare_amount") - col("prediction"))
).orderBy(
    col("error").desc()
).show(20, False)

+-----------+------------------+------------------+
|fare_amount|prediction        |error             |
+-----------+------------------+------------------+
|96.0       |55.8723937396327  |40.1276062603673  |
|73.0       |44.714094092361385|28.285905907638615|
|73.5       |48.601229034092   |24.898770965908   |
|71.5       |49.157328551504285|22.342671448495715|
|25.5       |47.722771907669944|22.222771907669944|
|6.5        |27.654815663559486|21.154815663559486|
|6.0        |26.366379534659085|20.366379534659085|
|80.0       |60.80198197492682 |19.198018025073182|
|69.0       |50.538802008366545|18.461197991633455|
|60.0       |41.994205789076304|18.005794210923696|
|73.5       |56.59014584047303 |16.909854159526972|
|75.0       |58.84540889114785 |16.154591108852152|
|35.0       |49.47969068140811 |14.479690681408108|
|26.5       |39.81878298030472 |13.318782980304718|
|50.0       |37.453142823095035|12.546857176904965|
|55.0       |43.913427986532234|11.086572013467766|
|40.5       

In [24]:
predictions.printSchema()

root
 |-- VendorID: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: integer (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_day: integer (nullable = true)
 |-- pickup_month: integer (nullab

#K-Means

In [25]:
feature_cols_cluster = [
    "trip_distance",
    "trip_duration",
    "fare_amount",
    "passenger_count"
]

In [27]:
assembler_cluster = VectorAssembler(
    inputCols=feature_cols_cluster,
    outputCol="features_cluster"
)

df_cluster = assembler_cluster.transform(df_ml)

In [30]:
kmeans = KMeans(
    featuresCol="features_cluster",
    predictionCol="cluster",
    k=3,
    seed=42
)

model_kmeans = kmeans.fit(df_cluster)

clusters = model_kmeans.transform(df_cluster)

In [36]:
evaluator = ClusteringEvaluator(
    predictionCol="cluster",
    featuresCol="features_cluster",
     metricName="silhouette"
)

silhouette = evaluator.evaluate(clusters)

print("Silhouette Score:", silhouette)

Silhouette Score: 0.7312151323417516


In [34]:
clusters.groupBy("cluster").count().show()

+-------+-----+
|cluster|count|
+-------+-----+
|      1|  689|
|      2| 3363|
|      0| 8125|
+-------+-----+



In [35]:
clusters.groupBy("cluster").avg(
    "trip_distance",
    "trip_duration",
    "fare_amount"
).show()

+-------+------------------+------------------+------------------+
|cluster|avg(trip_distance)|avg(trip_duration)|  avg(fare_amount)|
+-------+------------------+------------------+------------------+
|      1|13.931509433962264| 42.53957426221577| 44.01023222060958|
|      2| 4.416330657151352|21.105912379819607|18.112399643175735|
|      0| 1.394275692307692| 8.139751794871797|           7.62136|
+-------+------------------+------------------+------------------+

